# Notebook 2.0: Stratified Sampling & Temporal Coverage Audit

**Research Phase 2 — Data Ingestion & Sampling**
This notebook:
1. Discovers Parquet files in  across all target subreddits.
2. Extracts stratified samples by ISO week (~30,000 rows per file) to preserve temporal dynamics.
3. Audits timestamps to ensure real date coverage (2020–2025) and prevents the 1970 timestamp collapse (Bug 1 fix).
4. Generates a weekly volume coverage audit and saves .

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
nb_dir = Path.cwd()
project_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.modeling import set_seed
from src.ingest.sampling import get_parquet_from_path, stratified_sample_parquet_to_json
from src.modeling.load import load_samples, parse_created_utc

# Reproducibility seed
set_seed(42)

# Load configuration
config_path = project_root / 'config' / 'config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Loaded configuration for subreddits:', cfg['analysis']['subreddits'])

## Step 1: Discover Parquet Files
Discover all available parquet files in the raw data directory.

In [ ]:
raw_data_dir = (project_root / cfg['paths']['raw_data_dir']).resolve()
parquet_files = get_parquet_from_path(raw_data_dir)

print(f'Found {len(parquet_files)} parquet files:')
for pf in sorted(parquet_files):
    print('  -', Path(pf).name)

## Step 2: Stratified Sampling by Week
Sample rows stratified by ISO week of  into .

In [ ]:
samples_output_dir = project_root / 'data' / 'output' / 'samples' / '2.0'
desired_samples = cfg['analysis']['sample_size']

print(f'Extracting stratified samples ({desired_samples} per file) to: {samples_output_dir}')
stratified_sample_parquet_to_json(
    file_paths=parquet_files,
    output_dir=samples_output_dir,
    desired_samples=desired_samples,
    seed=cfg['modeling']['seed'],
)

## Step 3: Load Samples and Audit Timestamps
Verify that dates span the actual range (2020–2025) without collapsing to 1970-01-01 (Bug 1 fix).

In [ ]:
subreddits = cfg['analysis']['subreddits']
dataset_types = cfg['analysis']['dataset_types']

df_samples = load_samples(
    samples_dir=samples_output_dir,
    subreddits=subreddits,
    dataset_types=dataset_types,
)

print(f'Total records loaded: {len(df_samples):,}')
audit_table = df_samples.groupby(['subreddit', 'dataset_type'])['created_utc_dt'].agg(
    min_date='min',
    max_date='max',
    total_rows='count'
)
print(audit_table)

# Assert that timestamps are strictly post-2019
assert df_samples['created_utc_dt'].min().year >= 2020, 'Error: detected pre-2020 timestamp or 1970 bug!'
print('Timestamp validation passed: All timestamps are correctly parsed (2020-2025).')

## Step 4: Temporal Coverage Plot
Generate a weekly post volume plot across communities and highlight key event milestones.

In [ ]:
df_samples['week'] = df_samples['created_utc_dt'].dt.to_period('W').dt.start_time
weekly_volume = df_samples.groupby(['week', 'subreddit']).size().unstack(fill_value=0)

plt.figure(figsize=(14, 6))
for sub in weekly_volume.columns:
    plt.plot(weekly_volume.index, weekly_volume[sub], label=f'r/{sub}', alpha=0.8, linewidth=1.5)

# Milestones
plt.axvline(pd.Timestamp('2022-11-30'), color='red', linestyle='--', linewidth=2, label='ChatGPT Launch (Nov 30, 2022)')
plt.axvline(pd.Timestamp('2023-06-12'), color='black', linestyle=':', linewidth=1.5, label='Reddit API Blackout (June 2023)')

plt.title('Weekly Discourse Volume by Subreddit (Temporal Audit)', fontsize=14, pad=12)
plt.xlabel('Date (Weekly Bins)', fontsize=11)
plt.ylabel('Document Count (Sampled)', fontsize=11)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

fig_dir = project_root / 'data' / 'output' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / 'weekly_coverage_audit.png', dpi=300)
plt.show()

## Step 5: Export Coverage Audit Table
Export the temporal coverage table to .

In [ ]:
audit_csv_path = project_root / 'data' / 'output' / 'coverage_audit.csv'
weekly_volume.to_csv(audit_csv_path)
print(f'Coverage audit table saved successfully to: {audit_csv_path}')